# Classification using k-Nearest Neighbors and Evaluation
---

### Overview
- [First Toy Example on Decidion Boundaries](#First-Toy-Example-on-Decision-Boundaries)
- [Classifying Medical Data](#Classifying-Medical-Data)
  - [Data Understanding and Preprocessing](#Data-Understanding-and-Preprocessing)
  - [Classification using k-NN](#Classification-using-k-NN)
  - [Parameter Tuning using Cross-validation](#Parameter-Tuning-using-Cross-validation)
  - [Parameter Tuning using Grid Search and Cross-validation](#Parameter-Tuning-using-Grid-Search-and-Cross-validation)
- [Creating Pipelines](#Creating-Pipelines)  





In this notebook we will perform classification with the **k-nearest neighbors** algorithm. 

Of course we use the concepts which we studied before, but in this notebook we will also have a look at **hyperparameter tuning**. That means that we want to find a solid way to determine a good parameter $k$ for the algorithm. Here our imports:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

## First Toy Example on Decision Boundaries

This first toy example should illustrate the "overfitting" for k=1 and how using a larger k solves this problem. The idea of plotting the desision boundaries was adopted from [here](https://github.com/rhasanbd/K-Nearest-Neighbors-Learning-Without-Learning). First we create a (normalized) dataset, which is clearly not lineary separable:

In [ ]:
X, y = make_moons(n_samples=200, noise=0.4, random_state=42)
sc = MinMaxScaler()
X = sc.fit_transform(X) # normalize

plt.figure(figsize=(8, 4))
plt.plot(X[y.ravel()==0, 0], X[y.ravel()==0, 1], "bo", markeredgecolor="k")
plt.plot(X[y.ravel()==1, 0], X[y.ravel()==1, 1], "ro")

plt.title("2D Non-Linear Data Distribution", fontsize=16)
plt.xlabel("$x_1$", fontsize=14)
plt.ylabel("$x_2$", fontsize=14)
plt.show()

Now we split the data and train two k-NN models, one with k=1 and one with k=10. 

A helper function is used to plot the decision boundaries:

In [ ]:
def decision_boundary_class_colored(clf, X, plotDistanceFromHyperplane=False, colorBar=False):
    # Get the min and max value of feature x1
    x1min, x1max = X[:,0].min() - 0.1, X[:, 0].max() + 0.1
    
    # Get the min and max value of feature x2
    x2min, x2max = X[:,1].min() - 0.1, X[:, 1].max() + 0.1
    
    # Create the mesh grid
    x1s = np.linspace(x1min, x1max, 100)
    x2s = np.linspace(x2min, x2max, 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    
    # Create pairs of new points from the grid
    X_new = np.c_[x1.ravel(), x2.ravel()]
    
    # Compute the class predictions for all new points
    y_pred = clf.predict(X_new).reshape(x1.shape)
    
    # Generate the contourf plot for the predictions
    plt.contourf(x1, x2, y_pred, cmap=plt.cm.seismic, alpha=0.2)
    
    if(plotDistanceFromHyperplane == True):
        # Compute the signed distance of a sample to the hyperplane for all new points
        y_decision = clf.decision_function(X_new).reshape(x1.shape)

        # Generate the contourf plot for the distance of all points from the hyperplane
        plt.contourf(x1, x2, y_decision, cmap=plt.cm.bwr, alpha=0.2)
    
    if(colorBar==True):
        plt.colorbar()    

In [ ]:
# now the real code ...

# Split into train/test:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

knn_1 = KNeighborsClassifier(n_neighbors=1)
knn_10 = KNeighborsClassifier(n_neighbors=10)
knn_1.fit(X_train, y_train)
knn_10.fit(X_train, y_train)

print('Test accuracy with k=1:', knn_1.score(X_test, y_test))
print('Test accuracy with k=10:', knn_10.score(X_test, y_test))
    
# Show the decision boundary only along with training data
for m in [knn_1, knn_10]:
    plt.figure(figsize=(8,4))
    decision_boundary_class_colored(m, X_train,)
    plt.plot(X_train[y_train.ravel()==0, 0], X_train[y_train.ravel()==0, 1], "bo")
    plt.plot(X_train[y_train.ravel()==1, 0], X_train[y_train.ravel()==1, 1], "ro")
    plt.title(m , fontsize=16)
    plt.show()

Even if the test accuracy is the same for the two experiments, you could easily observe that the decision boundary is clearly overfitted for $k=1$.

## Classifying Medical Data

### Data Understanding and Preprocessing

The dataset we are using in the following is the "breast cancer Wisconsin" dataset, which contains data from 569 patients and  medical measuremnts of their tumor cells. A class label indicated whether a tumor is **malignant** or **benign**. 
Further information about the dataset can be found [here](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).

In [ ]:
breast_cancer = load_breast_cancer()

X = breast_cancer.data
y = breast_cancer.target

## Alternatively, load data from the internet ...
#df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data', header=None)
#X = df.loc[:, 2:].values # drop ID and class
#y = df.loc[:, 1].values
#le = LabelEncoder() # M,B -> 0,1
#y = le.fit_transform(y)

df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names) 
df['class'] = breast_cancer.target
print(df.shape)
df.head()

In [ ]:
#plt.bar(*np.unique(df['class'], return_counts=True))

import seaborn as sns
sns.countplot(x='class', data=df)
plt.show()

We can observe that the class prior is different. "0" indicates the malignant class, "1" the benign class. Finally, as a last data understanding step, let's plot the correlation of the (numeric) features:

In [ ]:
cor = df.corr()
sns.heatmap(cor, square = True, xticklabels=True, yticklabels=True) 
plt.show()

Interestingly, there are some attributes with a very high correlation, e.g. (worst|mean) perimeter and -area. It might be worth trying to remove them in a preprocessing step (we will not do that for now).

As a data preprocessing step, we will normalize the data:

In [ ]:
scaler = MinMaxScaler() # StandardScaler()
X_n = scaler.fit_transform(X)

And finally split the data into 80-20 train/test stratified:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_n, y, test_size=0.2, random_state=42, stratify=y)
print('X train shape: ', X_train.shape)
print('Y train shape: ', y_train.shape)
print('X test shape: ', X_test.shape)
print('Y test shape: ', y_test.shape)

Print the null-accuracy as a baseline (k-NN should not be worse than that):

In [ ]:
print('null accuracy: ',(y_test == 1).sum()/len(y_test))

The null accuracy is basically a classificator, which always predicts the class with the highest $P(C)$, in our case 1.

In other words: the null accuracy is identical to the class prior (c.f. Chapter 05) of the most probable class.

### Classification using k-NN

Let's try a one-nearest neighbor approach first and then set $k$ to a higher number to avoid overfitting...

In [ ]:
classifier = KNeighborsClassifier(n_neighbors = 1, metric = 'euclidean', weights='uniform')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy_score(y_test,y_pred)

In [ ]:
classifier = KNeighborsClassifier(n_neighbors = 10, metric = 'euclidean', weights='uniform')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy_score(y_test,y_pred)

In ~95% of the test cases we are able to do a correct prediction, not bad! Please keep in mind that k-NN does not directly take the class prior into account, however, checking the confusion matrix might be helpul as well:

In [ ]:
disp = ConfusionMatrixDisplay.from_estimator(
        classifier,
        X_test,
        y_test,
        cmap=plt.cm.Blues,
    )
plt.show()

...or check the f1-score \[or precision/recall\] instead of the accuracy:

Recap: f1 = 2 * (precision * recall) / (precision + recall)

In [ ]:
print(classification_report(y_test,y_pred))

Now we can try to find a good $k$ by trying out different values (please note, this is the **wrong way** as discussed in the lecture)

In [ ]:
accuracy_rate=[0.0]

for i in range(1,100):
    classifier = KNeighborsClassifier(n_neighbors=i, metric = 'euclidean', weights='uniform')
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    accuracy_rate.append(accuracy_score(y_test,y_pred))
#print(accuracy_rate)
print('Best k:',np.argmax(accuracy_rate),'@',np.max(accuracy_rate))

In our case, $k=5$ seems to deliver best results, but we tuned this parameter with the **test set**, which is not a fair evaluation. Furthermore, the accuracies seems pretty close to each other. To veryfiy, we will plot it:

In [ ]:
plt.plot(range(len(accuracy_rate)),accuracy_rate, color= 'green',linestyle = 'solid', marker='o', markersize=2)
plt.title('Accuracy vs. k-value')
plt.xlabel('k-value')
plt.ylabel('Accuracy')
plt.show()

Indeed, (besides cheating,) it might be just due to luck that we picked $k=5$ as a best result, many other $k$'s have similar (or even identical) accuracy. We will try to do better in the next section.. 

**Note:** "Better" in this case menas, that we will probably not be able to get the same accuracy as we did with cheating!

### Parameter Tuning using Cross-validation

We will perform now a 10-fold cross-validation (CV) for every single $k$ and compute the mean of the 10 results. 

This time we do not use the **test data** (no cheating - as it should be done) :

In [ ]:
accuracy_rate=[0.0]

for i in range(1,40): # only use 40 here since it gets worse > 40 anyway above and save some computation time...
    classifier = KNeighborsClassifier(n_neighbors=i, metric = 'euclidean', weights='uniform')
    score = cross_val_score(classifier, X_train, y_train, cv=10, n_jobs=-1) # n_jobs=X to run on X cores in parallel.
    accuracy_rate.append(score.mean())
print('Best k:',np.argmax(accuracy_rate),'@',np.max(accuracy_rate))

In [ ]:
plt.plot(range(len(accuracy_rate)),accuracy_rate, color= 'green',linestyle = 'solid', marker='o', markersize=2)
plt.title('Accuracy vs. k-value')
plt.xlabel('k-value')
plt.ylabel('Accuracy')
plt.show()

We can observe that the best $k$ is now $7$ and $5$ wold be a (slightly) worse choice in this situation. It is also remarkable that there seems to be no big difference among $5 \leq k < 20$ and the curve looks now much smoother, because the accuracy is *averaged* over the 10 runs from every cross validation!

In short: statistically, $k=7$ is the best choice in a fair evaluation, if we are not cheating.

Please read the details about the parameters of the [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html). You will find out that it uses in our case stratified sampling (good!) and also uses the same folds for all of our $k$ values (also good!).

We still have to try $k=7$ using the test data and hope, that it is close to our optimal, but "cheated" result:

In [ ]:
classifier = KNeighborsClassifier(n_neighbors = 7, metric = 'euclidean', weights='uniform')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy_score(y_test,y_pred)

In this case we are lucky and get the same result as with cheating. As mentioned earlier, a worse result was expected.

**Note:** The used dataset here is not very big. A single instance in the test set is already changing ~1% of accuracy. The results here depend strongly also on the chance, how the test set was choosen -- to this end, maybe another crossvalidation over the complete dataset would give as more reliable "testing accuracy". This would also be called a **nested cross validation**.

### Parameter Tuning using Grid Search and Cross-validation

If you have enough computation power and/or time, we can also try to tune more hyperparameters. The `GridSearchCV` helps us with that by trying out all different combinations of (multiple) parameters in order to find the best set. Of course, we use a cross validation each time to make sure the hyperparameters are really meaningful, not just a "lucky choice". Also, we completely **hold out** the test set, as it should be done!

In [ ]:
param_grid = {'n_neighbors': np.arange(1,40, 2), 
              'weights': ['uniform', 'distance'],
              'metric' : ['euclidean','manhattan']}
classifier = KNeighborsClassifier()

knn_cv = GridSearchCV(classifier, param_grid, scoring='accuracy', cv=10, verbose=3, n_jobs=-1)
knn_cv.fit(X_train, y_train)

In [ ]:
params_optimal_knn = knn_cv.best_params_

print("Best Score: %f" % knn_cv.best_score_)
print("Optimal Hyperparameter Values: ", params_optimal_knn)
print("\n")


Now let's use these "best" parameters, train our classifier and apply it on the test set:

In [ ]:
classifier = KNeighborsClassifier(metric ='euclidean', n_neighbors= 7, weights= 'uniform')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# The three lines above, meaning training the "best" classifier on the full training set can also be done
# by GridSearchCV. In this case add "refit=True" as a parameter in the GridSearchCV. This causes GridSearchCV at the 
# end of the evaluation to use the *full* trainingdata and the best parameters to train the best model.
# Prediction is than done with this "best" model.
# This would require than this line only:
# y_pred = knn_cv.best_estimator_.predict(X_test)

accuracy_score(y_test,y_pred)

~98% accuracy on the test set seems like a good result (compare to the results reported on the [website](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic); note -- these numbers are CV results). Interestingly, the "euclidean" distance and the uniform parameter for the neighbor influence was choosen as best parameter (we intuitively set them right).

## Creating Pipelines

Pipelines are a method in `scikit-learn` how to quickly connect and try different steps of the CRISP-DM (data preparation, modeling, evaluation) with only a few lines of code. Sometimes, they are very handy to quickly put things together, but however they violate the concept of "**storytelling**" in jupyter notebooks. 

Also, data understanding is skipped. Since they are quite helpful, there will be a short introduction here, but please try to avoid using them generally "out of the box", e.g. in your project.

To further improve our k-NN, we can try to apply a PCA before the classification or use a different normalization. Additionally, we might also try a differnt classifier.

In [ ]:
# No scaling...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

# MinMax with PCA
pipe_pca = make_pipeline(MinMaxScaler(),
                        PCA(n_components=10),
                        KNeighborsClassifier(n_neighbors = 7, metric = 'euclidean', weights='uniform'))

pipe_pca.fit(X_train, y_train)
y_pred = pipe_pca.predict(X_test)
test_acc = pipe_pca.score(X_test, y_test)
print(f'With PCA test accuracy: {test_acc:.3f}')

# StandardScaler
pipe_std = make_pipeline(StandardScaler(),
                        KNeighborsClassifier(n_neighbors = 7, metric = 'euclidean', weights='uniform'))

pipe_std.fit(X_train, y_train)
y_pred = pipe_std.predict(X_test)
test_acc = pipe_std.score(X_test, y_test)
print(f'Standardization test accuracy: {test_acc:.3f}')

# GaussianNB as classifier
from sklearn.naive_bayes import GaussianNB
pipe_nb = make_pipeline(MinMaxScaler(),
                        GaussianNB())

pipe_nb.fit(X_train, y_train)
y_pred = pipe_nb.predict(X_test)
test_acc = pipe_nb.score(X_test, y_test)
print(f'Naive Bayes test accuracy: {test_acc:.3f}')

# LogisticRegression as classifier (just another "linear" model/classifier, more on that in lecture 9)
from sklearn.linear_model import LogisticRegression
pipe_lr = make_pipeline(MinMaxScaler(),
                        LogisticRegression())

pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)
test_acc = pipe_lr.score(X_test, y_test)
print(f'Logistic Regression test accuracy: {test_acc:.3f}')


The [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) function of `scikit-learn` provides a concatenation of *transformers* (objects that *fit -transform*) followed by an *Estimator* (*prediction model*). In the first example, we use two transformers: `MinMaxScaler` and `PCA`, followed by a k-NN as a prediction model. When using a pipeline, the `fit` and `transform` functions of the transformers are called in sequence and finally the prediction model is *fitted*. The entire pipeline thus serves as a *meta-prediction model*.

A big advantage is also that the pipeline can be used directly with other datasets and this way also for CV!

Finally, we want to make use of this and also use a `GridSearchCV` with the pipeline!

In [ ]:
# please be patient...
pipe_pca = make_pipeline(MinMaxScaler(),
                        PCA(),
                        KNeighborsClassifier())

param_grid = {'pca__n_components' : np.arange(2,30),
              'kneighborsclassifier__n_neighbors': np.arange(1,40, 2), 
              'kneighborsclassifier__weights': ['uniform', 'distance'],
              'kneighborsclassifier__metric' : ['euclidean','manhattan']}


gs = GridSearchCV(estimator=pipe_pca, 
                  param_grid=param_grid, 
                  scoring='accuracy', 
                  refit=True, # neccessary to use inside pipeline
                  cv=10,
                  n_jobs=-1)
gs = gs.fit(X_train, y_train)
print(gs.best_score_)
print(gs.best_params_)

In [ ]:
pipe_pca.fit(X_train, y_train)
y_pred = pipe_pca.predict(X_test)
test_acc = pipe_pca.score(X_test, y_test)
print(f'With PCA test accuracy: {test_acc:.3f}')

Finally, test (and train) accuracy did not improve significantly, however, we could show that the dimensionality reduction using PCA for this dataset makes sense (31 -> 11 dimensions) -- probably because there are correlated features in the data as the correlation plot already showed us in the beginning.